# FOI – 230724012 – Number of Mothers/Fathers applied for a CAO application in Wales

## 1. Import packages and set options 
<a name="import_packages"></a>

In [1]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import xlsxwriter
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [2]:
#this is the database we will be extracting from
database = "familyman_dev_v3"

#this extracts the February snapshot from athena
snapshot_date = "2023-05-10"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 1
pub_year = 2023

## *Check: FCSQ*

In [4]:
# Making sure the number of C100 orders applied for match the CSV Children National 2023 Q1
    # Filter by type = Application, count_type = Order type, and order_type_code = (29 or 30 or 31 or 32)

# The fcsq.ca_apps_order_types removes the child ID and groups up so we only count an order type within an event once, rather than one per child

check = pydb.read_sql_query("SELECT year, count(*) as n  FROM fcsq.ca_apps_order_types WHERE year > 2010 and (order_code = 29 OR order_code = 30 OR order_code = 31 OR order_code = 32) group by year order by year")
check

,year,n
0,2011,67867
1,2012,71827
2,2013,72060
3,2014,54674
4,2015,55720
5,2016,62260
6,2017,67450
7,2018,71624
8,2019,78993
9,2020,85191


## *Check: The total number of orders applied for by applicant and respondent RTC (Private Law PowerBi Dashboard)*

In [6]:
# Checking the number of private law orders applied for grouped by order types in a C100 application
check = pydb.read_sql_query("SELECT year, order_case_type, order_code, count(*) as n FROM fcsq.ca_apps_order_types WHERE order_case_type = 'P' and (order_code = 29 or order_code = 30 or order_code = 31 or order_code = 32) GROUP BY year, order_case_type, order_code")
check

,year,order_case_type,order_code,n
0,2009,P,29,33978
1,2016,P,32,7356
2,2005,P,30,19131
3,2015,P,30,15783
4,2002,P,31,5945
...,...,...,...,...
105,1998,P,30,140
106,2018,P,31,11127
107,2007,P,31,7882
108,2020,P,32,10930


## Extract relationship to child info from Familyman

In [19]:
#Extract applicant and respondent relationship to child information
pydb.create_temp_table( 
f"""
SELECT    
  value AS rtc,
  field_model,
  role
FROM 
  {database}.role_fields
WHERE
  field_model IN ('APLC_RC','RSPC_RC')
  AND mojap_snapshot_date = DATE'{snapshot_date}'
""",

"app_resp_rtc")

In [20]:
app_resp_rtc_data = pydb.read_sql_query("SELECT * FROM __temp__.app_resp_rtc")
app_resp_rtc_data

,rtc,field_model,role
0,Mother,RSPC_RC,7819312
1,Mother,RSPC_RC,7822116
2,Mother,RSPC_RC,7823534
3,Mother,APLC_RC,7823799
4,Father,APLC_RC,7827232
...,...,...,...
2359710,Mother,RSPC_RC,8448925
2359711,Other,APLC_RC,8451629
2359712,Mother,RSPC_RC,8451764
2359713,Father,RSPC_RC,8459013


## RTC - Applicants

In [77]:
#Get rtc for applicants
#As joining applicants using role id it will automatically filter out and rspc_rc records from the app_resp_rtc table
pydb.create_temp_table( 
f"""
SELECT    
  a.case_number,
  a.year,
  a.event,
  CAST(NULL AS INTEGER) AS quarter,
  a.order_case_type,
  CASE WHEN a.order_code IN (35,36) THEN 36
       ELSE a.order_code END
     AS order_code,
  CASE WHEN a.order_code IN (35,36) THEN 'Enforcement (inc ammendments and breaches)'
        WHEN a.order_code = 37 THEN 'Enforcement - revocation'
        ELSE a.Order_desc END 
     AS Order_desc,   
  ap.role_id,
  r.field_model AS app_resp,
  CASE WHEN (r.rtc IN ('--','Child')
              OR r.rtc IS NULL)
        THEN 'Not recorded'
        ELSE r.rtc
    END AS rtc,
    a.court_code, 
    a.app_dfj, 
    a.app_region

FROM 
  fcsq.ca_apps_order_types a
  LEFT JOIN fcsq.ca_applicants ap 
    ON a.case_number = ap.case_number
  LEFT JOIN __temp__.app_resp_rtc r
    ON ap.role_id = r.role

""",

"apps_rtc")

In [78]:
check = pydb.read_sql_query("SELECT * FROM __temp__.apps_rtc where case_number = 'CF21P70621'")
check

,case_number,year,event,quarter,order_case_type,order_code,order_desc,role_id,app_resp,rtc,court_code,app_dfj,app_region
0,CF21P70621,2021,16402019070,<NA>,P,31,Section 8 Prohibited Steps Order,12636593,APLC_RC,Father,164,South East Wales DFJ,WALES
1,CF21P70621,2021,16402019070,<NA>,P,32,Section 8 Specific Issue Order,12636593,APLC_RC,Father,164,South East Wales DFJ,WALES
2,CF21P70621,2022,16402111324,<NA>,P,30,Section 8 Residence Order,12636593,APLC_RC,Father,164,South East Wales DFJ,WALES
3,CF21P70621,2021,16402019070,<NA>,P,29,Section 8 Contact Order,12636593,APLC_RC,Father,164,South East Wales DFJ,WALES
4,CF21P70621,2021,16402019070,<NA>,P,30,Section 8 Residence Order,12636593,APLC_RC,Father,164,South East Wales DFJ,WALES


## *Check: Number of Mother/Father Applicants (Private Law PowerBi Dashboard)*

In [79]:
# Checking the number of private law orders applied for grouped by order types in a C100 application
check = pydb.read_sql_query("SELECT year, order_case_type, order_code, app_resp, rtc, count(*) as n FROM __temp__.apps_rtc WHERE order_case_type = 'P' and app_resp = 'APLC_RC' and (order_code = 29 or order_code = 30 or order_code = 31 or order_code = 32) and (rtc = 'Mother' or rtc = 'Father') and year > 2011 GROUP BY year, order_case_type, order_code, app_resp, rtc")
check

,year,order_case_type,order_code,app_resp,rtc,n
0,2023,P,32,APLC_RC,Father,1318
1,2012,P,29,APLC_RC,Father,21387
2,2019,P,31,APLC_RC,Mother,6561
3,2015,P,31,APLC_RC,Mother,4651
4,2015,P,30,APLC_RC,Mother,6099
...,...,...,...,...,...,...
91,2021,P,31,APLC_RC,Father,6343
92,2019,P,30,APLC_RC,Mother,9231
93,2019,P,32,APLC_RC,Mother,5503
94,2019,P,29,APLC_RC,Father,22786


## RTC - Respondents

In [80]:
#Get rtc for resps
#As joining respondents using role id it will automatically filter out and aplc_rc records from the app_resp_rtc table
pydb.create_temp_table( 
f"""
SELECT    
  a.case_number,
  a.year,
  a.event,
  CAST(NULL AS INTEGER) AS quarter,
  a.order_case_type,
  CASE WHEN a.order_code IN (35,36) THEN 36
       ELSE a.order_code END
     AS order_code,
  CASE WHEN a.order_code IN (35,36) THEN 'Enforcement (inc ammendments and breaches)'
        WHEN a.order_code = 37 THEN 'Enforcement - revocation'
        ELSE a.Order_desc END 
     AS Order_desc,  
  rp.role_id,
  r.field_model AS app_resp,
  CASE WHEN (r.rtc IN ('--','Child')
              OR r.rtc IS NULL)
        THEN 'Not recorded'
        ELSE r.rtc
    END AS rtc,
    a.court_code, 
    a.app_dfj, 
    a.app_region
    
    
FROM 
  fcsq.ca_apps_order_types a
  LEFT JOIN fcsq.ca_respondents rp 
    ON a.case_number = rp.case_number
  LEFT JOIN __temp__.app_resp_rtc r
    ON rp.role_id = r.role
""",

"resps_rtc")

In [81]:
check = pydb.read_sql_query("SELECT * FROM __temp__.resps_rtc where case_number = 'CF21P70621'")
check

,case_number,year,event,quarter,order_case_type,order_code,order_desc,role_id,app_resp,rtc,court_code,app_dfj,app_region
0,CF21P70621,2021,16402019070,<NA>,P,32,Section 8 Specific Issue Order,12636575,RSPC_RC,Mother,164,South East Wales DFJ,WALES
1,CF21P70621,2021,16402019070,<NA>,P,31,Section 8 Prohibited Steps Order,12636575,RSPC_RC,Mother,164,South East Wales DFJ,WALES
2,CF21P70621,2022,16402111324,<NA>,P,30,Section 8 Residence Order,12636575,RSPC_RC,Mother,164,South East Wales DFJ,WALES
3,CF21P70621,2021,16402019070,<NA>,P,29,Section 8 Contact Order,12636575,RSPC_RC,Mother,164,South East Wales DFJ,WALES
4,CF21P70621,2021,16402019070,<NA>,P,30,Section 8 Residence Order,12636575,RSPC_RC,Mother,164,South East Wales DFJ,WALES


## *Check: Number of Mother/Father Respondents (Private Law PowerBi Dashboard)*

In [82]:
# Checking the number of private law orders applied for grouped by order types in a C100 application
check = pydb.read_sql_query("SELECT year, order_case_type, order_code, app_resp, rtc, count(*) as n FROM __temp__.resps_rtc WHERE order_case_type = 'P' and app_resp = 'RSPC_RC' and (order_code = 29 or order_code = 30 or order_code = 31 or order_code = 32) and (rtc = 'Mother' or rtc = 'Father') and year > 2011 GROUP BY year, order_case_type, order_code, app_resp, rtc")
check

,year,order_case_type,order_code,app_resp,rtc,n
0,2018,P,30,RSPC_RC,Father,7195
1,2014,P,31,RSPC_RC,Father,3728
2,2015,P,29,RSPC_RC,Mother,15211
3,2013,P,30,RSPC_RC,Father,8659
4,2020,P,31,RSPC_RC,Mother,4930
...,...,...,...,...,...,...
91,2014,P,30,RSPC_RC,Father,5969
92,2018,P,32,RSPC_RC,Mother,3269
93,2017,P,31,RSPC_RC,Mother,3860
94,2018,P,30,RSPC_RC,Mother,9383


## Matching Applicants to Respondents - filter by 2012 < year < 2023

In [84]:
pydb.create_temp_table( 
f"""
  SELECT DISTINCT a.case_number,
  a.year,
  a.event,
  a.order_case_type,
  a.order_code,
  a.order_desc,  
  a.rtc as app_rtc,
  b.rtc as resp_rtc,
  a.court_code, 
  a.app_dfj, 
  a.app_region
  
  FROM
       __temp__.apps_rtc a 
    INNER JOIN __temp__.resps_rtc b
     on a.case_number = b.case_number and a.event = b.event 
  
  WHERE a.year > 2012 and a.year < 2023

""",

"app_resp_match")

In [85]:
count = pydb.read_sql_query("SELECT count(*) as n FROM __temp__.app_resp_match")
count

,n
0,1174310


In [86]:
check = pydb.read_sql_query("SELECT * FROM __temp__.app_resp_match")
check

,case_number,year,event,order_case_type,order_code,order_desc,app_rtc,resp_rtc,court_code,app_dfj,app_region
0,RL22P00004,2022,30800848712,P,30,Section 8 Residence Order,Mother,Father,308,Wrexham DFJ,WALES
1,LS13P08110,2013,24303011221,P,31,Section 8 Prohibited Steps Order,Father,Mother,243,Leeds DFJ,NORTH EAST
2,CV17P00585,2017,18002150243,P,31,Section 8 Prohibited Steps Order,Mother,Father,180,Coventry DFJ,MIDLANDS
3,LV20P02132,2020,25105161607,P,30,Section 8 Residence Order,Father,Mother,251,Liverpool DFJ,NORTH WEST
4,BS21P70886,2022,15103053769,P,27,Special Guardianship Order,Other,Mother,151,Bristol DFJ,SOUTH WEST
...,...,...,...,...,...,...,...,...,...,...,...
1174305,SE22P00084,2022,32003194899,P,29,Section 8 Contact Order,Other,Mother,320,Sheffield DFJ,NORTH EAST
1174306,ZE22P00840,2022,12101345623,P,29,Section 8 Contact Order,Father,Mother,121,East London Family Court,LONDON
1174307,MA22P01334,2022,26206589429,P,31,Section 8 Prohibited Steps Order,Father,Not recorded,262,Manchester DFJ,NORTH WEST
1174308,ZW22P01524,2022,33200988120,P,32,Section 8 Specific Issue Order,Father,Mother,332,West London Family Court,LONDON


## Wales and C100 only figures 

In [87]:
pydb.create_temp_table( 
f"""
  SELECT *
  
  FROM
       __temp__.app_resp_match
  
  WHERE app_region = 'WALES'
  AND (order_code = 29 or order_code = 30 or order_code = 31 or order_code = 32)
  AND ((app_rtc = 'Mother' and resp_rtc = 'Father') OR (resp_rtc = 'Mother' AND app_rtc = 'Father'))

""",

"app_resp_match_wales_only")

In [88]:
check = pydb.read_sql_query("SELECT * FROM __temp__.app_resp_match_wales_only")
check

,case_number,year,event,order_case_type,order_code,order_desc,app_rtc,resp_rtc,court_code,app_dfj,app_region
0,WX16P00601,2016,38400603329,P,29,Section 8 Contact Order,Mother,Father,384,Wrexham DFJ,WALES
1,NP14P00432,2014,28000940939,P,29,Section 8 Contact Order,Father,Mother,280,South East Wales DFJ,WALES
2,CF21P00292,2021,29900774180,P,30,Section 8 Residence Order,Father,Mother,299,South East Wales DFJ,WALES
3,CJ20P00229,2020,15900587349,P,30,Section 8 Residence Order,Father,Mother,159,Caernarfon DFJ,WALES
4,SA16P01329,2016,21700291173,P,32,Section 8 Specific Issue Order,Mother,Father,217,Swansea DFJ,WALES
...,...,...,...,...,...,...,...,...,...,...,...
34165,GM13P00683,2013,58500219313,P,29,Section 8 Contact Order,Father,Mother,585,Cardiff DFJ,WALES
34166,SA20P01127,2020,34401598271,P,31,Section 8 Prohibited Steps Order,Mother,Father,344,Swansea DFJ,WALES
34167,NP21P70006,2021,28002157518,P,30,Section 8 Residence Order,Father,Mother,280,South East Wales DFJ,WALES
34168,SA21P00260,2021,27400763748,P,29,Section 8 Contact Order,Mother,Father,274,Swansea DFJ,WALES


## *Check: Number of Applications made per Case Number > 1*

In [99]:
check = pydb.read_sql_query("SELECT * FROM (SELECT case_number, count(*) as n FROM __temp__.app_resp_match_wales_only WHERE app_rtc = 'Father' GROUP BY CASE_NUMBER ORDER BY n) WHERE n > 1")
check

,case_number,n
0,CF22P00063,2
1,CJ21P00058,2
2,NP17P00575,2
3,RL22P00035,2
4,NP22P70119,2
...,...,...
5122,SA21P00268,5
5123,SA21P00194,5
5124,NP19P00920,5
5125,CF21P70621,5


## Final Output: Applications

In [102]:
mother_app = pydb.read_sql_query("SELECT year, count(*) as n FROM (SELECT DISTINCT year, case_number, event, app_region, app_rtc, resp_rtc FROM __temp__.app_resp_match_wales_only GROUP BY year, case_number, event, app_region, app_rtc, resp_rtc) WHERE app_rtc = 'Mother' and resp_rtc = 'Father' GROUP BY year ORDER BY year")
mother_app

,year,n
0,2013,758
1,2014,541
2,2015,562
3,2016,690
4,2017,748
5,2018,901
6,2019,979
7,2020,945
8,2021,930
9,2022,1017


In [103]:
father_app = pydb.read_sql_query("SELECT year, count(*) as n FROM (SELECT DISTINCT year, case_number, event, app_region, app_rtc, resp_rtc FROM __temp__.app_resp_match_wales_only GROUP BY year, case_number, event, app_region, app_rtc, resp_rtc) WHERE app_rtc = 'Father' and resp_rtc = 'Mother' GROUP BY year ORDER BY year")
father_app

,year,n
0,2013,1927
1,2014,1351
2,2015,1443
3,2016,1599
4,2017,1654
5,2018,1665
6,2019,1824
7,2020,1626
8,2021,1684
9,2022,1563


## Create Final Output

In [104]:
#creating a panda dataframe for both order and applicant tables
#uses the queries underneath both order and applicant table
df1 = pd.DataFrame(mother_app)

df2 = pd.DataFrame(father_app)

In [105]:
#creates a seperate sheet for each dataframe 
with io.BytesIO() as xlsx_data:
    with pd.ExcelWriter(xlsx_data) as writer:
        
        df1.to_excel(writer, sheet_name='mother_app', index=False)
        
        df2.to_excel(writer, sheet_name='father_app', index=False)
        
        
    ## Upload the in-memory data to S3
    s3 = boto3.resource("s3")
    bucket = s3.Bucket("alpha-family-data")
    object_name = "FOIs/FOI 230724012 – working table.xlsx"
    s3.Bucket("alpha-family-data").put_object(Key=object_name, Body=xlsx_data.getvalue())